In [ ]:
# ============================================================
# ANN_PI Hyperparameter Grid Search
# Mean and standard deviation of performance across 50 seeds
# Station-level 70:30 training-validation split
#
# Author: Junyoung Lee
# Affiliation: Ulsan National Institute of Science and Technology (UNIST)
# Email: junyounglee@unist.ac.kr
# ============================================================

import os
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# Repository paths
# ------------------------------------------------------------
# This notebook can be run from either the repository root
# or the notebooks directory.
current_dir = Path.cwd()
project_dir = (
    current_dir.parent
    if current_dir.name == 'notebooks'
    else current_dir
)

data_dir = project_dir / 'data'
output_dir = project_dir / 'results'
output_dir.mkdir(parents=True, exist_ok=True)

input_file = data_dir / 'Total data_for submission.csv'
total_data = pd.read_csv(input_file)
)

# Use the absolute value of U.ratio
total_data['U.ratio'] = np.abs(total_data['U.ratio'])

station_col = 'SSN'
seeds = list(range(1, 51))


# -----------------------------
# Feature sets
# -----------------------------
feature_sets = {
    'Set1': ['U.ratio', 'Mw', 'EpiD', 'E.Dep', 'ray.p', 'j.angle'],
    'Set2': ['U.ratio', 'Mw', 'EpiD', 'E.Dep', 'ray.p', 'j.angle', 'S.Lat', 'S.Long'],
    'Set3': ['U.ratio', 'Mw', 'EpiD', 'E.Dep', 'ray.p', 'j.angle', 'slope_500m'],
    'Set4': ['U.ratio', 'Mw', 'EpiD', 'E.Dep', 'ray.p', 'j.angle', 'S.Lat', 'S.Long', 'slope_500m']
}


# -----------------------------
# Hyperparameter grid
# -----------------------------
hidden_layer_sizes_list = [
    (50, 50)
]

learning_rate_init_list = [
    0.0001,
    0.0005,
    0.001,
    0.005,
    0.01,
    0.05,
    0.1,
    0.2
]

alpha_list = [
    0.001,
    0.01,
    0.1,
    0.5
]


# -----------------------------
# Prepare the PI target
# -----------------------------
total_data = total_data[
    (total_data['Vs30_mea'] > 0) &
    (total_data['Vs30_f0'] > 0)
].copy()

total_data['log_res'] = (
    np.log(total_data['Vs30_mea']) -
    np.log(total_data['Vs30_f0'])
)


# Columns shared by all feature sets
all_features = sorted(
    set(sum(feature_sets.values(), []))
)

# Remove missing values using common columns so that all models
# are compared using the same samples
required_cols = (
    [station_col, 'Vs30_mea', 'Vs30_f0', 'log_res'] +
    all_features
)

total_data = (
    total_data[required_cols]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .copy()
)


# -----------------------------
# Grid search
# -----------------------------
summary_rows = []

total_combinations = (
    len(hidden_layer_sizes_list) *
    len(learning_rate_init_list) *
    len(alpha_list)
)

combo_count = 0

for hidden_layer_sizes, learning_rate_init, alpha in product(
    hidden_layer_sizes_list,
    learning_rate_init_list,
    alpha_list
):

    combo_count += 1

    print(
        f'Running {combo_count}/{total_combinations}: '
        f'hidden_layer_sizes={hidden_layer_sizes}, '
        f'learning_rate_init={learning_rate_init}, '
        f'alpha={alpha}'
    )

    seed_results = []

    for seed in seeds:

        # Generate a reproducible station-level split
        np.random.seed(seed)

        stations = np.array(
            sorted(total_data[station_col].unique())
        )

        np.random.shuffle(stations)

        n_train = int(len(stations) * 0.7)

        train_stations = stations[:n_train]
        test_stations = stations[n_train:]

        train_data = total_data[
            total_data[station_col].isin(train_stations)
        ].copy()

        test_data = total_data[
            total_data[station_col].isin(test_stations)
        ].copy()

        y_test = test_data['Vs30_mea'].values
        y_pred_pwave = test_data['Vs30_f0'].values

        row = {
            'seed': seed,
            'RMSE_P-wave': np.sqrt(
                mean_squared_error(
                    y_test,
                    y_pred_pwave
                )
            ),
            'R2_P-wave': r2_score(
                y_test,
                y_pred_pwave
            )
        }

        for set_name, features in feature_sets.items():

            X_train = train_data[features]
            y_train = train_data['log_res']

            X_test = test_data[features]

            model = Pipeline([
                (
                    'scaler',
                    StandardScaler()
                ),
                (
                    'ann',
                    MLPRegressor(
                        hidden_layer_sizes=hidden_layer_sizes,
                        activation='relu',
                        solver='adam',
                        alpha=alpha,
                        learning_rate_init=learning_rate_init,
                        max_iter=4000,
                        early_stopping=True,
                        validation_fraction=0.15,
                        random_state=seed
                    )
                )
            ])

            model.fit(
                X_train,
                y_train
            )

            predicted_log_residual = model.predict(
                X_test
            )

            # Physics-informed Vs30 prediction
            y_pred = (
                test_data['Vs30_f0'].values *
                np.exp(predicted_log_residual)
            )

            y_true = test_data['Vs30_mea'].values

            row[f'RMSE_{set_name}'] = np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred
                )
            )

            row[f'R2_{set_name}'] = r2_score(
                y_true,
                y_pred
            )

        seed_results.append(row)

    seed_results_df = pd.DataFrame(seed_results)

    mean_row = {
        'hidden_layer_sizes': str(hidden_layer_sizes),
        'alpha': alpha,
        'learning_rate_init': learning_rate_init,

        'RMSE_P-wave': (
            seed_results_df['RMSE_P-wave'].mean()
        ),
        'RMSE_Set1': (
            seed_results_df['RMSE_Set1'].mean()
        ),
        'RMSE_Set2': (
            seed_results_df['RMSE_Set2'].mean()
        ),
        'RMSE_Set3': (
            seed_results_df['RMSE_Set3'].mean()
        ),
        'RMSE_Set4': (
            seed_results_df['RMSE_Set4'].mean()
        ),

        'R2_P-wave': (
            seed_results_df['R2_P-wave'].mean()
        ),
        'R2_Set1': (
            seed_results_df['R2_Set1'].mean()
        ),
        'R2_Set2': (
            seed_results_df['R2_Set2'].mean()
        ),
        'R2_Set3': (
            seed_results_df['R2_Set3'].mean()
        ),
        'R2_Set4': (
            seed_results_df['R2_Set4'].mean()
        )
    }

    summary_rows.append(mean_row)


# -----------------------------
# Save the summary
# -----------------------------
summary_df = pd.DataFrame(summary_rows)

summary_df = summary_df[
    [
        'hidden_layer_sizes',
        'alpha',
        'learning_rate_init',
        'RMSE_P-wave',
        'RMSE_Set1',
        'RMSE_Set2',
        'RMSE_Set3',
        'RMSE_Set4',
        'R2_P-wave',
        'R2_Set1',
        'R2_Set2',
        'R2_Set3',
        'R2_Set4'
    ]
]

output_file = os.path.join(
    output_dir,
    'ANN_PI_hyperparameter_summary_mean_modi-layer_sizes_50 - 3_layers.csv'
)

summary_df.to_csv(
    output_file,
    index=False
)

print('\nSaved:', output_file)
print(summary_df.head())